# MNIST odd vs. even — weight-activation interpretability

Train a model with `scripts/train.py` (or a separate training notebook), then
load it here by setting `EXPERIMENT_DIR` below.  All architecture, dataset, and
analysis parameters are read from the saved `config.json` — no other edits
needed to switch between experiments.

## 0. Setup

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src import (
    collect_activations,
    compute_synaptic_arbors,
    nmf_component_sweep, full_nmf_pipeline, reconstruct_from_components,
    plot_nmf_scree, plot_nmf_component, plot_factor_graph,
    plot_neuron_nmf_component, plot_neuron_nmf_scatter,
    single_neuron_report,
    load_experiment, get_transform, get_loaders_from_config,
    evaluate,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Load experiment

Change `EXPERIMENT_DIR` to point at any directory created by `scripts/train.py`.
Everything else is derived from `config.json` automatically.

In [ ]:
# ── the only line you need to change when switching experiments ──
#EXPERIMENT_DIR = '../experiments/mnist_even_odd_mlp_20_10'
#EXPERIMENT_DIR = '../experiments/mnist_even_odd_mlp_40_20_10'
EXPERIMENT_DIR = '../experiments/mnist_even_odd_mlp_10_5'

model, cfg = load_experiment(EXPERIMENT_DIR, device)
train_loader, test_loader = get_loaders_from_config(cfg)
label_transform = get_transform(cfg['label_transform'])

LAYER_INDICES = cfg['analysis_layer_indices']
INPUT_SIDE    = cfg['input_side']
N_PER_CLASS   = cfg['n_per_class']

print(cfg['description'])
print(model)

In [ ]:
# ── sanity-check: verify the loaded checkpoint is actually trained ──────────
import torch.nn as nn
_crit = nn.CrossEntropyLoss()
_, _acc = evaluate(model, test_loader, criterion=_crit,
                   label_transform=label_transform, device=device)
print(f'Test accuracy: {_acc:.4f}')
if _acc < 0.85:
    print('WARNING: accuracy < 85% — model may not be trained.'
          '  Run scripts/train.py first.')
else:
    print('Model accuracy looks good.')

In [ ]:
import math

def _nice_shape(n):
    """Largest near-square (H, W) factorisation of n for imshow."""
    h = round(math.sqrt(n))
    while h > 1 and n % h != 0:
        h -= 1
    return (h, n // h)

# ── Architecture-derived — auto-computed from model, never edit these ─────────
linear_indices      = model.linear_layer_indices()
LAYER_SIZES         = [model.layers[li].out_features for li in linear_indices]
PRUNABLE_LAYERS     = linear_indices              # all linear layers including pixel→L1
act_offsets         = np.cumsum([0] + LAYER_SIZES[:-1])
# imshow shape for inputs to each non-first linear layer
# in_shapes_per_layer[0] = display shape for L2 inputs (= L1 output)
# in_shapes_per_layer[i] = display shape for layer (i+2) inputs
in_shapes_per_layer = [_nice_shape(LAYER_SIZES[i]) for i in range(len(LAYER_SIZES) - 1)]

# ── Per-experiment analysis parameters — change these when switching models ───
CLASS_NAMES  = {0: 'even', 1: 'odd'}   # {class_id: display_name}
DIGIT_SUBSET = [0, 1]                   # digits to highlight in mean-arbor plots
K_GLOBAL     = 5                        # NMF components for Section 3 (global)
# One K per linear layer (PRUNABLE_LAYERS[i] = linear_indices[i]).
# Default: 5 per layer.  Tune per-layer after reviewing scree plots.
K_PER_LAYER_HIDDEN = [5] * len(PRUNABLE_LAYERS)
# Example override: K_PER_LAYER_HIDDEN = [5, 10, 5, 3]

print(f'LAYER_SIZES:         {LAYER_SIZES}')
print(f'PRUNABLE_LAYERS:     {PRUNABLE_LAYERS}')
print(f'in_shapes_per_layer: {in_shapes_per_layer}')
print(f'K_PER_LAYER_HIDDEN:  {K_PER_LAYER_HIDDEN}')

## 2. Collect activations

Capture intermediate layer outputs at the indices stored in `config.json`
for correctly classified training samples (up to `n_per_class` per class).

In [ ]:
acts = collect_activations(
    model,
    train_loader.dataset,
    layer_indices=LAYER_INDICES,
    label_transform=label_transform,
    n_per_class=N_PER_CLASS,
    device=device,
)

for cl in acts['classes']:
    print(f"class {cl}: {acts['by_class'][cl].shape}")

# convenient shorthands
all_acts    = acts['all']       # (total_n, n_neurons)
all_targets = acts['targets']   # (total_n,)  task labels (e.g. 0=even, 1=odd)
all_images  = acts['images']    # (total_n, C, H, W)
all_digits  = acts['digits']    # (total_n,)  original dataset labels (e.g. 0-9)
classes     = acts['classes']

### 2b. Layer-wise activation statistics

In [ ]:
# Derive LAYER_SIZES / act_offsets here so this cell is self-contained
_lin_idx    = model.linear_layer_indices()
LAYER_SIZES = [model.layers[li].out_features for li in _lin_idx]
act_offsets = np.cumsum([0] + LAYER_SIZES[:-1])
layer_names = [f'L{i+1} ({s} neurons)' for i, s in enumerate(LAYER_SIZES)]

hdr = f"{'Layer':<22}  {'mean':>7}  {'std':>7}  {'sparsity':>10}"
print(hdr)
print('-' * len(hdr))
for i, (name, size) in enumerate(zip(layer_names, LAYER_SIZES)):
    col = all_acts[:, act_offsets[i]:act_offsets[i] + size]
    print(f'{name:<22}  {col.mean():7.4f}  {col.std():7.4f}  '
          f'{(col < 0.05).mean():9.2%}')

## 3. NMF decomposition per class

Factorize each class's joint activation matrix to find interpretable neural
"factors" — groups of neurons that co-activate and together explain the
classification of that class.

In [ ]:
# Sweep over candidate component counts; 3 random seeds each.
K_RANGE = range(1, 16)

for cl in classes:
    print(f'\n── Scree: class {cl} ──')
    scree = nmf_component_sweep(acts['by_class'][cl], n_components_range=K_RANGE)
    fig = plot_nmf_scree(scree)
    fig.suptitle(f'Class {cl}')
    plt.show()


In [ ]:
N_COMPONENTS = K_GLOBAL

nmf_results = {}
for cl in classes:
    X = acts['by_class'][cl]
    img_f, neural_f, lams = full_nmf_pipeline(X, N_COMPONENTS)
    nmf_results[cl] = dict(img_factors=img_f, neural_factors=neural_f, lambdas=lams)
    print(f'class {cl}  lambdas: {lams}')
    print(f'  img_factors: {img_f.shape}  neural_factors: {neural_f.shape}')

### 3a. Component visualizations

In [ ]:
first_layer_weights = model.layers[linear_indices[0]].weight.detach().numpy()

for cl in classes:
    img_factors    = nmf_results[cl]['img_factors']
    neural_factors = nmf_results[cl]['neural_factors']
    class_mask   = all_targets == cl
    class_images = all_images[class_mask, 0]
    class_digits = all_digits[class_mask]
    print(f'\n── Component visualizations: class {cl} ──')
    for fi in range(N_COMPONENTS):
        fig = plot_nmf_component(
            fi, neural_factors, img_factors, LAYER_SIZES,
            class_images, class_digits, first_layer_weights,
            image_side=INPUT_SIDE,
        )
        plt.show()

### 3b. Factor graphs (network scaffolding)

In [ ]:
for cl in classes:
    img_factors    = nmf_results[cl]['img_factors']
    neural_factors = nmf_results[cl]['neural_factors']
    print(f'\n── Factor graphs: class {cl} ──')
    for fi in range(N_COMPONENTS):
        fig = plot_factor_graph(fi, neural_factors, model.layers, LAYER_SIZES, linear_indices,
                                img_factors=img_factors)
        plt.show()


### 3c. Cross-class NMF alignment

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

nf0 = nmf_results[0]['neural_factors']   # (32, K)
nf1 = nmf_results[1]['neural_factors']
sim = cosine_similarity(nf0.T, nf1.T)   # (K, K)

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(sim, vmin=0, vmax=1, cmap='viridis', aspect='auto')
ax.set(xlabel='class 1 (odd) component', ylabel='class 0 (even) component',
       title='Cross-class NMF alignment\n(cosine similarity of neural factors)',
       xticks=range(N_COMPONENTS), yticks=range(N_COMPONENTS))
plt.colorbar(im, ax=ax, label='cosine sim')
fig.tight_layout(); plt.show()

### 3d. NMF component stability across seeds

In [ ]:
from scipy.optimize import linear_sum_assignment

N_STAB_SEEDS = 5
for cl in classes:
    X = acts['by_class'][cl]
    _, nf_ref, _ = full_nmf_pipeline(X, N_COMPONENTS, random_state=0)
    sims = []
    for seed in range(1, N_STAB_SEEDS):
        _, nf_s, _ = full_nmf_pipeline(X, N_COMPONENTS, random_state=seed)
        C = cosine_similarity(nf_ref.T, nf_s.T)   # (K, K)
        row, col = linear_sum_assignment(-C)        # Hungarian matching
        sims.append(C[row, col].mean())
    print(f'class {cl}: stability = {np.mean(sims):.4f} ± {np.std(sims):.4f}  '
          f'(mean matched cosine sim, {N_STAB_SEEDS-1} seed pairs)')

## 4. Single-neuron decoding analysis

For each neuron: compute synaptic arbors (weight × input), project with PCA,
and show mean arbors grouped by task class and by selected digit labels.

**Layer offsets in `all_acts`** are determined by `LAYER_SIZES` and `LAYER_INDICES`.
For the default model (20→10→2 with post-sigmoid capture):
- cols `0 : LAYER_SIZES[0]` = layer-1 activations (inputs to linear layer 2)
- cols `LAYER_SIZES[0] : LAYER_SIZES[0]+LAYER_SIZES[1]` = layer-2 activations

`input_shape` controls how the weight/arbor vector is reshaped for imshow:
- layer 1: `28` → (28, 28) pixels
- layer 2: `(4, 5)` → 20 layer-1 activations
- layer 3: `(2, 5)` → 10 layer-2 activations

In [ ]:
# CLASS_NAMES, DIGIT_SUBSET, act_offsets are set near the top of the notebook.
# Inputs to each non-first linear layer = activations of the preceding layer.
# Inputs to the first linear layer = flattened pixel values.
layer_inputs = {}
layer_inputs[linear_indices[0]] = all_images[:, 0].reshape(len(all_images), -1)
for lii in range(1, len(linear_indices)):
    li = linear_indices[lii]
    layer_inputs[li] = all_acts[:, act_offsets[lii - 1]:act_offsets[lii]]

In [ ]:
### Layer 1 neurons — pixel-level arbors (28×28)

LI = linear_indices[0]   # e.g. model.layers[1]

for local_ni in [9, 7]:
    global_ni = act_offsets[0] + local_ni
    print(f'─── layer 1, neuron {local_ni} ───')
    figs = single_neuron_report(
        model.layers, LI, local_ni,
        all_acts, all_images[:, 0],        # pixel arbors
        all_targets, classes, INPUT_SIDE,  # int → reshape (28, 28)
        class_names=CLASS_NAMES,
        digit_targets=all_digits,
        digit_subset=DIGIT_SUBSET,
        global_neuron_idx=global_ni,
    )
    for fig in figs:
        plt.show()

In [ ]:
### Layer 2 neurons — arbors over layer-1 activations

LI = linear_indices[1]
l2_inputs   = layer_inputs[LI]
l2_in_shape = in_shapes_per_layer[0]   # auto-derived from LAYER_SIZES[0]

for local_ni in range(min(3, LAYER_SIZES[1])):   # first 3 neurons of layer 2
    global_ni = act_offsets[1] + local_ni
    print(f'─── layer 2, neuron {local_ni} ───')
    figs = single_neuron_report(
        model.layers, LI, local_ni,
        all_acts, None,
        all_targets, classes, l2_in_shape,
        class_names=CLASS_NAMES,
        digit_targets=all_digits,
        digit_subset=DIGIT_SUBSET,
        inputs=l2_inputs,
        global_neuron_idx=global_ni,
    )
    for fig in figs:
        plt.show()

In [ ]:
### Layer 3 neurons — arbors over layer-2 activations

LI = linear_indices[2]   # e.g. model.layers[5]
l3_inputs   = layer_inputs[LI]
l3_in_shape = in_shapes_per_layer[1]   # auto-derived from LAYER_SIZES[1]

for local_ni in range(LAYER_SIZES[2]):   # all output neurons (usually 2)
    global_ni = act_offsets[2] + local_ni
    print(f'─── layer 3, neuron {local_ni} ───')
    figs = single_neuron_report(
        model.layers, LI, local_ni,
        all_acts, None,
        all_targets, classes, l3_in_shape,
        class_names=CLASS_NAMES,
        digit_targets=all_digits,
        digit_subset=DIGIT_SUBSET,
        inputs=l3_inputs,
        global_neuron_idx=global_ni,
    )
    for fig in figs:
        plt.show()
# For deeper models, add similar cells for linear_indices[3], etc.

### 4b. Neuron pairwise correlation matrix

In [ ]:
corr = np.corrcoef(all_acts.T)   # (n_neurons, n_neurons)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
for boundary in act_offsets[1:]:
    ax.axhline(boundary - 0.5, color='k', lw=1)
    ax.axvline(boundary - 0.5, color='k', lw=1)
tick_pos = [int(act_offsets[i] + LAYER_SIZES[i] / 2) for i in range(len(LAYER_SIZES))]
tick_lbl = [f'L{i+1}' for i in range(len(LAYER_SIZES))]
ax.set(title='Neuron pairwise Pearson correlation',
       xlabel='neuron index', ylabel='neuron index',
       xticks=tick_pos, xticklabels=tick_lbl,
       yticks=tick_pos, yticklabels=tick_lbl)
plt.colorbar(im, ax=ax, label='Pearson r')
fig.tight_layout(); plt.show()

## TODO / future directions

- Show negative weight × activation products
- Weight products by the neuron's actual activation fraction
- Optimize node positions using similarity metrics (e.g. cosine of arbor vectors)
- Tensor NMF (NTF): use input pixels directly instead of averaging
- Identify hub neurons; test effect of silencing them
- Find maximally-activating images for each factor
- Try outlier digits and verify the factor graph changes substantially
- Repeat analysis for a different task (10-way digit ID) or a deeper model

## 5. Per-neuron NMF analysis

Section 3 factorizes the joint activation matrix across **all** neurons to find
groups that co-activate.  This section instead zooms in on a **single neuron**
and factorizes its synaptic arbor matrix — weight × input for every sample:

```
arbors (n_samples, n_inputs) ≈ img_factors (n_samples, K) × arbor_factors.T (K, n_inputs)
```

- `arbor_factors[:, k]` — the k-th prototypical input pattern that drives this
  neuron (a data-driven receptive-field basis element).
- `img_factors[:, k]` — how strongly each image activates that pattern.

NMF requires non-negative input, so arbors are clipped to positive values
before factorization.

In [ ]:
import math

def _nice_shape(n):
    """Largest near-square (H, W) factorization of n for imshow."""
    h = round(math.sqrt(n))
    while h > 1 and n % h != 0:
        h -= 1
    return (h, n // h)

# ── neuron selection — only these two lines need changing ────────────────────
NMF_LAYER_IDX = linear_indices[0]   # any value from linear_indices
NMF_LOCAL_NI  = 0                    # local neuron index within that layer

# ── everything below is derived automatically ────────────────────────────────
_lii          = linear_indices.index(NMF_LAYER_IDX)
NMF_GLOBAL_NI = int(act_offsets[_lii]) + NMF_LOCAL_NI

if _lii == 0:                                            # layer 1: pixel arbors
    nmf_arb_inputs  = all_images[:, 0].reshape(len(all_images), -1)
    nmf_input_shape = INPUT_SIDE                         # int → (28, 28) in imshow
else:                                                    # deeper layers: act arbors
    nmf_arb_inputs  = layer_inputs[NMF_LAYER_IDX]
    nmf_input_shape = _nice_shape(nmf_arb_inputs.shape[1])

ws         = model.layers[NMF_LAYER_IDX].weight[NMF_LOCAL_NI].detach().numpy()
arbors     = compute_synaptic_arbors(ws, nmf_arb_inputs)
arbors_pos = np.clip(arbors, 0, None)

print(f'neuron: layer {NMF_LAYER_IDX}, local idx {NMF_LOCAL_NI}, '
      f'global idx {NMF_GLOBAL_NI}')
print(f'arbors shape: {arbors_pos.shape},  '
      f'input display shape: {nmf_input_shape},  '
      f'non-zero fraction: {(arbors_pos > 0).mean():.2%}')

In [ ]:
# Scree: explained variance vs K for this neuron's arbors
K_RANGE_NEURON = range(1, 12)
scree_n = nmf_component_sweep(arbors_pos, n_components_range=K_RANGE_NEURON)
fig = plot_nmf_scree(scree_n)
fig.suptitle(f'Per-neuron scree — layer {NMF_LAYER_IDX}, neuron {NMF_LOCAL_NI}')
plt.show()

In [ ]:
# ── set K based on the elbow above ──
N_COMP_NEURON = 10

In [ ]:
img_f, arb_f, lams = full_nmf_pipeline(arbors_pos, N_COMP_NEURON)
print('lambdas:', lams)

for fi in range(N_COMP_NEURON):
    fig = plot_neuron_nmf_component(
        fi, img_f, arb_f, nmf_input_shape,
        all_images[:, 0], all_targets, classes,
        class_names=CLASS_NAMES,
        digit_targets=all_digits,
    )
    plt.show()

In [ ]:
# 2D component scatter — do the NMF coefficients separate classes?
fig = plot_neuron_nmf_scatter(
    img_f, all_targets, classes,
    fi=0, fj=1,
    class_names=CLASS_NAMES,
    digit_targets=all_digits,
)
plt.show()

## 6. Stimulus-conditioned scaffold graphs

Per-neuron NMF is run for **every** neuron in layers 2 and 3 (silently — no
per-neuron plots).  The per-stimulus NMF coefficients `img_f[s, k]` are then
used to build class/digit/custom-conditioned scaffold graphs.

**Node loading for a selected stimulus subset S:**
- *Layer 1* (no NMF): mean activation over S — `mean_{s∈S}(all_acts[s, n])`
- *Layers 2 & 3*: λ-weighted mean coefficient — `Σ_k λ_k · mean_{s∈S}(img_f_n[s, k])`

This loading is fed as a single-column `neural_factors` into `plot_factor_graph`,
so edge thickness reflects how strongly each neuron drives the network for the
selected stimuli.

In [ ]:
# NMF components per hidden layer — set at top of notebook in K_PER_LAYER_HIDDEN
# K_PER_LAYER_HIDDEN[i] applies to PRUNABLE_LAYERS[i] = linear_indices[i+1]
print(f'Per-layer K: {list(zip(PRUNABLE_LAYERS, K_PER_LAYER_HIDDEN))}')

In [ ]:
# ── Per-neuron NMF for all prunable layers (positive and negative arbors) ────
# PRUNABLE_LAYERS now includes L1 (pixel→L1 weights); layer_inputs[li] returns
# pixel values for L1 and neuron activations for deeper layers.
nmf_pos = {}   # {li: (img_f_list, arb_f_list, lams_list)}
nmf_neg = {}

for lii_idx, li in enumerate(PRUNABLE_LAYERS):
    k   = K_PER_LAYER_HIDDEN[lii_idx]
    W   = model.layers[li].weight.detach().numpy()
    inp = layer_inputs[li]
    ip, ap, lp = [], [], []
    im, am, lm = [], [], []
    for ni in range(W.shape[0]):
        arbs = compute_synaptic_arbors(W[ni], inp)
        imf, arf, lam = full_nmf_pipeline(np.clip(arbs,  0, None), k)
        ip.append(imf); ap.append(arf); lp.append(lam)
        imf, arf, lam = full_nmf_pipeline(np.clip(-arbs, 0, None), k)
        im.append(imf); am.append(arf); lm.append(lam)
    nmf_pos[li] = (ip, ap, lp)
    nmf_neg[li] = (im, am, lm)
    n_in = W.shape[1]
    print(f'layer {li} (L{lii_idx+1}): {W.shape[0]} neurons × {k} pos + {k} neg components  '
          f'[{W.shape[0]}×{n_in} weights]')

In [ ]:
from src import compute_effective_arbors, plot_scaffold_graph

def make_scaffold_graph(mask, title=''):
    loading   = all_acts[mask].mean(axis=0)

    # ── Neuron→neuron scaffold (L2+ layers) ──────────────────────────────────
    pos_edges = [compute_effective_arbors(*nmf_pos[li], mask) for li in PRUNABLE_LAYERS[1:]]
    neg_edges = [compute_effective_arbors(*nmf_neg[li], mask) for li in PRUNABLE_LAYERS[1:]]
    fig = plot_scaffold_graph(loading, pos_edges, LAYER_SIZES, neg_edge_matrices=neg_edges)
    fig.suptitle(title, fontsize=10)

    # ── Pixel→L1 effective arbors ─────────────────────────────────────────────
    li0   = PRUNABLE_LAYERS[0]
    E_pos = compute_effective_arbors(*nmf_pos[li0], mask)   # (n_L1, 784)
    E_neg = compute_effective_arbors(*nmf_neg[li0], mask)
    E_l1  = E_pos - E_neg                                   # net (signed) arbor
    n_l1  = LAYER_SIZES[0]
    l1_load = loading[:n_l1]

    fig2, axes = plt.subplots(1, n_l1, figsize=(2 * n_l1, 2.2))
    if n_l1 == 1:
        axes = [axes]
    for ni in range(n_l1):
        arbor  = E_l1[ni].reshape(INPUT_SIDE, -1)
        absmax = np.abs(arbor).max() or 1
        axes[ni].imshow(arbor, vmin=-absmax, vmax=absmax, cmap='seismic')
        axes[ni].set(title=f'L1-n{ni}\nload={l1_load[ni]:.2f}', xticks=[], yticks=[])
    fig2.suptitle(f'{title} — L1 pixel arbors', fontsize=9)
    fig2.tight_layout()

    return fig, fig2

In [ ]:
# ── 2 graphs: one per task class (even / odd) ────────────────────────────────
for cl, cl_name in CLASS_NAMES.items():
    mask = all_targets == cl
    fig, fig2 = make_scaffold_graph(mask, title=f'scaffold — {cl_name} (n={mask.sum()})')
    plt.show()
    plt.show()

In [ ]:
# ── 10 graphs: one per digit ─────────────────────────────────────────────────
for d in sorted(np.unique(all_digits).tolist()):
    mask = all_digits == d
    fig, fig2 = make_scaffold_graph(mask, title=f'scaffold — digit {d} (n={mask.sum()})')
    plt.show()
    plt.show()

In [ ]:
# ── custom stimulus selection ─────────────────────────────────────────────────
# Edit CUSTOM_INDICES to select any subset of samples by their position in
# all_acts / all_targets / all_images.  Use np.where(all_digits == 3)[0][:10]
# to pick the first 10 samples of digit 3, for example.
CUSTOM_INDICES = [0, 5, 12, 47]

mask = np.zeros(len(all_targets), dtype=bool)
mask[CUSTOM_INDICES] = True
fig, fig2 = make_scaffold_graph(mask, title=f'scaffold — custom ({mask.sum()} stimuli)')
plt.show()
plt.show()

## 7. Pruning analysis — is the scaffold the critical path?

The scaffold graph assigns an importance score to each L1→L2 and L2→L3 weight.
If those scores correctly identify the *critical path*, then:

- Removing **highest-importance** weights first → accuracy collapses fastest
- Removing **lowest-importance** weights first → accuracy degrades slowest
- Removing **random** weights → intermediate degradation (baseline)

Only the 220 weights covered by the scaffold (layers 2 and 3) are pruned;
layer-1 pixel→neuron weights remain intact throughout.

In [ ]:
# ── Strategy A: weight magnitude ─────────────────────────────────────────────
imp_mag = np.concatenate([
    np.abs(model.layers[li].weight.detach().numpy()).flatten()
    for li in PRUNABLE_LAYERS
])

# ── Strategy E: class-differential scaffold |E_even − E_odd| ─────────────────
mask_even = all_targets == classes[0]
mask_odd  = all_targets == classes[1]

def _eff_layer(li, mask):
    return (compute_effective_arbors(*nmf_pos[li], mask) +
            compute_effective_arbors(*nmf_neg[li], mask))

E_even = {li: _eff_layer(li, mask_even) for li in PRUNABLE_LAYERS}
E_odd  = {li: _eff_layer(li, mask_odd)  for li in PRUNABLE_LAYERS}
imp_diff = np.concatenate([np.abs(E_even[li] - E_odd[li]).flatten() for li in PRUNABLE_LAYERS])

print(f'prunable weights: {len(imp_mag)}  '
      f'|W| range: [{imp_mag.min():.4f}, {imp_mag.max():.4f}]  '
      f'diff range: [{imp_diff.min():.4f}, {imp_diff.max():.4f}]')

In [ ]:
# ── Strategy B: weight × mean input activation ────────────────────────────────
# layer_inputs[li] holds pixel values for L1 and neuron activations for L2+,
# so this formula is uniform across all layers.
imp_act  = np.concatenate([
    np.abs(model.layers[li].weight.detach().numpy() *
           layer_inputs[li].mean(axis=0)[None, :]
           ).flatten()
    for lii_idx, li in enumerate(PRUNABLE_LAYERS)
])

# ── Strategy C: global NMF neuron loading (Section 3) broadcast ──────────────
nf0, lam0 = nmf_results[classes[0]]['neural_factors'], nmf_results[classes[0]]['lambdas']
nf1, lam1 = nmf_results[classes[1]]['neural_factors'], nmf_results[classes[1]]['lambdas']
neuron_load = (nf0 * lam0).sum(axis=1) + (nf1 * lam1).sum(axis=1)   # (n_neurons_total,)
# lii_idx now starts at 0 = L1, so output neurons are at act_offsets[lii_idx].
# in_features gives the correct input count (784 for L1, LAYER_SIZES[i-1] otherwise).
imp_global_nmf = np.concatenate([
    np.outer(
        neuron_load[act_offsets[lii_idx]:act_offsets[lii_idx] + LAYER_SIZES[lii_idx]],
        np.ones(model.layers[li].in_features)
    ).flatten()
    for lii_idx, li in enumerate(PRUNABLE_LAYERS)
])

# ── Strategy D: per-neuron NMF total effective arbor (E_even + E_odd) ────────
imp_total = np.concatenate([(E_even[li] + E_odd[li]).flatten() for li in PRUNABLE_LAYERS])

for name, imp in [('act-scaled',  imp_act),
                  ('global-nmf',  imp_global_nmf),
                  ('nmf-total',   imp_total)]:
    print(f'{name:15s} range: [{imp.min():.4f}, {imp.max():.4f}]')

In [ ]:
import torch.nn as nn

W_orig   = {li: model.layers[li].weight.data.clone() for li in PRUNABLE_LAYERS}
W_numel  = [model.layers[li].weight.numel() for li in PRUNABLE_LAYERS]
W_bounds = np.concatenate([[0], np.cumsum(W_numel)]).astype(np.int64)
n_prunable_weights = int(W_bounds[-1])
criterion = nn.CrossEntropyLoss()

def eval_with_pruning(prune_indices):
    prune_indices = np.asarray(prune_indices, dtype=np.int64)
    with torch.no_grad():
        for idx, li in enumerate(PRUNABLE_LAYERS):
            lo, hi = int(W_bounds[idx]), int(W_bounds[idx + 1])
            layer_idx = prune_indices[(prune_indices >= lo) & (prune_indices < hi)] - lo
            W_p = W_orig[li].clone()
            W_p.view(-1)[layer_idx] = 0.0
            model.layers[li].weight.data.copy_(W_p)
        acc = evaluate(model, test_loader, criterion=criterion,
                       label_transform=label_transform, device=device)[1]
        for li in PRUNABLE_LAYERS:
            model.layers[li].weight.data.copy_(W_orig[li])
    return acc

print('unpruned accuracy:', eval_with_pruning([]))

In [ ]:
PRUNE_FRACTIONS = np.linspace(0, 1, 21)
N_RAND_SEEDS    = 10
rng = np.random.default_rng(0)

all_strategies = {
    'magnitude':   imp_mag,
    'act-scaled':  imp_act,
    'global-nmf':  imp_global_nmf,
    'nmf-total':   imp_total,
    'nmf-diff':    imp_diff,
}
results = {f'{name}_{d}': [] for name in all_strategies for d in ('high', 'low')}
results.update({'random_mean': [], 'random_lo': [], 'random_hi': []})

for frac in PRUNE_FRACTIONS:
    n_prune = int(round(frac * n_prunable_weights))
    for name, imp in all_strategies.items():
        results[f'{name}_high'].append(eval_with_pruning(np.argsort(imp)[::-1][:n_prune]))
        results[f'{name}_low'].append( eval_with_pruning(np.argsort(imp)[:n_prune]))
    rand_accs = [eval_with_pruning(rng.permutation(n_prunable_weights)[:n_prune])
                 for _ in range(N_RAND_SEEDS)]
    results['random_mean'].append(float(np.mean(rand_accs)))
    results['random_lo'].append(float(np.min(rand_accs)))
    results['random_hi'].append(float(np.max(rand_accs)))
print('sweep complete')

In [ ]:
STYLE = {
    'magnitude':   ('C0', '-'),
    'act-scaled':  ('C1', '-'),
    'global-nmf':  ('C2', '-'),
    'nmf-total':   ('C3', '-'),
    'nmf-diff':    ('C4', '-'),
}
pct = PRUNE_FRACTIONS * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, direction, title in [
    (axes[0], 'high', 'High-importance-first (critical path)'),
    (axes[1], 'low',  'Low-importance-first (redundant-first)'),
]:
    ax.fill_between(pct, results['random_lo'], results['random_hi'],
                    color='gray', alpha=0.2)
    ax.plot(pct, results['random_mean'], 'k-', lw=1.5,
            label=f'random (n={N_RAND_SEEDS})')
    for name, (color, ls) in STYLE.items():
        ax.plot(pct, results[f'{name}_{direction}'],
                color=color, ls=ls, lw=2, label=name)
    ax.set(xlabel='weights pruned (%)', ylabel='test accuracy', title=title)
    ax.legend(fontsize=7)
fig.suptitle('Pruning comparison — all strategies, L2 & L3 weights', fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
# K-robustness sweep: re-run each NMF strategy for a range of K values to
# test whether conclusions depend on the (heuristic) number of components.
# K_LAYER_RANGES[i]: [K-1, K, K+1] for PRUNABLE_LAYERS[i] (parallel, not a grid)
K_LAYER_RANGES = [[k - 1, k, k + 1] for k in K_PER_LAYER_HIDDEN]
K_GLOBAL_RANGE = [K_GLOBAL - 1, K_GLOBAL, K_GLOBAL + 1]

k_curves = {s: {d: [] for d in ('high', 'low')}
            for s in ('nmf-total', 'nmf-diff', 'global-nmf')}

def _eff_layer_k(li, mask, p, n):
    return compute_effective_arbors(*p[li], mask) + compute_effective_arbors(*n[li], mask)

# ── Global NMF robustness ─────────────────────────────────────────────────────
for k_g in K_GLOBAL_RANGE:
    _, nf_ev, lam_ev = full_nmf_pipeline(acts['by_class'][classes[0]], k_g)
    _, nf_od, lam_od = full_nmf_pipeline(acts['by_class'][classes[1]], k_g)
    nl = (nf_ev * lam_ev).sum(axis=1) + (nf_od * lam_od).sum(axis=1)
    imp_g_k = np.concatenate([
        np.outer(
            nl[act_offsets[lii_idx]:act_offsets[lii_idx] + LAYER_SIZES[lii_idx]],
            np.ones(model.layers[li].in_features)
        ).flatten()
        for lii_idx, li in enumerate(PRUNABLE_LAYERS)
    ])
    h, lo = [], []
    for frac in PRUNE_FRACTIONS:
        n_p = int(round(frac * n_prunable_weights))
        h.append(eval_with_pruning(np.argsort(imp_g_k)[::-1][:n_p]))
        lo.append(eval_with_pruning(np.argsort(imp_g_k)[:n_p]))
    k_curves['global-nmf']['high'].append(h); k_curves['global-nmf']['low'].append(lo)
    print(f'global-nmf K={k_g} done')

# ── Per-neuron NMF robustness ─────────────────────────────────────────────────
n_K_combos = len(K_LAYER_RANGES[0])   # 3 (K-1, K, K+1)
for ki in range(n_K_combos):
    nmf_pos_k = {}; nmf_neg_k = {}
    for lii_idx, li in enumerate(PRUNABLE_LAYERS):
        k   = K_LAYER_RANGES[lii_idx][ki]
        W   = model.layers[li].weight.detach().numpy()
        inp = layer_inputs[li]
        ip, ap, lp = [], [], []
        im, am, lm = [], [], []
        for ni in range(W.shape[0]):
            arbs = compute_synaptic_arbors(W[ni], inp)
            imf, arf, lam = full_nmf_pipeline(np.clip(arbs,  0, None), k)
            ip.append(imf); ap.append(arf); lp.append(lam)
            imf, arf, lam = full_nmf_pipeline(np.clip(-arbs, 0, None), k)
            im.append(imf); am.append(arf); lm.append(lam)
        nmf_pos_k[li] = (ip, ap, lp); nmf_neg_k[li] = (im, am, lm)
    Ee_k = {li: _eff_layer_k(li, mask_even, nmf_pos_k, nmf_neg_k) for li in PRUNABLE_LAYERS}
    Eo_k = {li: _eff_layer_k(li, mask_odd,  nmf_pos_k, nmf_neg_k) for li in PRUNABLE_LAYERS}
    k_imps = {
        'nmf-total': np.concatenate([(Ee_k[li] + Eo_k[li]).flatten() for li in PRUNABLE_LAYERS]),
        'nmf-diff':  np.concatenate([np.abs(Ee_k[li] - Eo_k[li]).flatten() for li in PRUNABLE_LAYERS]),
    }
    for sname, imp_k in k_imps.items():
        h, lo = [], []
        for frac in PRUNE_FRACTIONS:
            n_p = int(round(frac * n_prunable_weights))
            h.append(eval_with_pruning(np.argsort(imp_k)[::-1][:n_p]))
            lo.append(eval_with_pruning(np.argsort(imp_k)[:n_p]))
        k_curves[sname]['high'].append(h); k_curves[sname]['low'].append(lo)
    ks = [K_LAYER_RANGES[j][ki] for j in range(len(PRUNABLE_LAYERS))]
    print(f'per-neuron NMF K={ks} done')

In [ ]:
ROBUST_STYLE = {
    'global-nmf': ('C2', f'K∈{K_GLOBAL_RANGE}'),
    'nmf-total':  ('C3', f'K∈{K_LAYER_RANGES}'),
    'nmf-diff':   ('C4', f'K∈{K_LAYER_RANGES}'),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, direction, title in [
    (axes[0], 'high', 'High-importance-first'),
    (axes[1], 'low',  'Low-importance-first'),
]:
    ax.fill_between(pct, results['random_lo'], results['random_hi'],
                    color='gray', alpha=0.2, label='random range')
    ax.plot(pct, results['random_mean'], 'k-', lw=1.5, label='random mean')
    for name, color in [('magnitude', 'C0'), ('act-scaled', 'C1')]:
        ax.plot(pct, results[f'{name}_{direction}'], color=color, lw=2, label=name)
    for sname, (color, klabel) in ROBUST_STYLE.items():
        mat = np.array(k_curves[sname][direction])   # (n_K, 21)
        ax.fill_between(pct, mat.min(0), mat.max(0), color=color, alpha=0.2)
        ax.plot(pct, mat.mean(0), color=color, lw=2,
                label=f'{sname} (mean±range, {klabel})')
    ax.set(xlabel='weights pruned (%)', ylabel='test accuracy', title=title)
    ax.legend(fontsize=7)
fig.suptitle('K-robustness — NMF strategies vs deterministic baselines', fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
baseline_acc = eval_with_pruning([])
sens = {}
for lii_idx, li in enumerate(PRUNABLE_LAYERS):
    n_out = model.layers[li].out_features
    n_in  = model.layers[li].in_features
    node_offset = int(act_offsets[lii_idx])
    for local_n in range(n_out):
        flat_start = int(W_bounds[lii_idx]) + local_n * n_in
        idx = np.arange(flat_start, flat_start + n_in)
        label = f'L{lii_idx + 1}-n{local_n} (node {node_offset + local_n})'
        sens[label] = baseline_acc - eval_with_pruning(idx)

print(f'Baseline accuracy: {baseline_acc:.4f}')
print('Neuron sensitivity (accuracy drop when zeroed):')
for k, v in sorted(sens.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:+.4f}')

In [ ]:
# Keep a subset of the first hidden-to-hidden layer (PRUNABLE_LAYERS[1] = L2) active.
# Edit KEEP_HIDDEN1_NEURONS based on scaffold inspection.
KEEP_HIDDEN1_NEURONS = [5,7]    # local indices within L2 (first hidden layer)
KEEP_INPUT_NEURONS   = None      # None = keep all inputs; or e.g. [0, 13]

n_h1_out = model.layers[PRUNABLE_LAYERS[1]].out_features
n_h1_in  = model.layers[PRUNABLE_LAYERS[1]].in_features
# W_bounds index for PRUNABLE_LAYERS[1]
h1_bound_start = int(W_bounds[1])
prune_idx = []
for local_n in range(n_h1_out):
    row_start = local_n * n_h1_in
    if local_n not in KEEP_HIDDEN1_NEURONS:
        prune_idx.extend(range(h1_bound_start + row_start,
                               h1_bound_start + row_start + n_h1_in))
    elif KEEP_INPUT_NEURONS is not None:
        keep = set(h1_bound_start + row_start + c for c in KEEP_INPUT_NEURONS)
        prune_idx.extend(
            set(range(h1_bound_start + row_start,
                      h1_bound_start + row_start + n_h1_in)) - keep
        )

acc_sub = eval_with_pruning(np.array(prune_idx, dtype=np.int64))
print(f'Subgraph (L2 neurons {KEEP_HIDDEN1_NEURONS}): acc = {acc_sub:.4f}  '
      f'(baseline {baseline_acc:.4f}, chance ~{1/len(classes):.2f})')